# An-Ra V4 — Canonical Shared-Folder T4 Trainer

This notebook continues the canonical 181M-parameter V4 model from the latest verified full-resume checkpoint. It uses one canonical writer, a signed launch contract, deterministic token windows, and Drive-backed checkpoint durability every 200 optimizer steps or 60 minutes.

**Before Run all:** select a T4 GPU runtime. Share the single `ANRA_T4_TRAINING_HOME` folder with **Editor** access and add its shortcut to the Colab account's My Drive. The folder directly contains the latest full-resume checkpoint, both immutable data-pack parts, this notebook, and the campaign signing identity. The notebook refuses to mix assets from other Drive locations. Do not run two notebooks with `WORKER_ROLE = "canonical_trainer"` at the same time.

In [ ]:
# Operator configuration
WORKER_ROLE = "canonical_trainer"  # canonical_trainer or verify_only
WORKER_ID = "colab-t4-primary"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
REPO_REF = "iterate500"
SESSION_BUDGET_MINUTES = 180
DRAIN_RESERVE_MINUTES = 30
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8

DRIVE_MOUNT_ROOT = "/content/drive"
PACK_PARTS = [
    ("v4_phase_a_170m_seed1301.tar.gz.part00", 83886080, "9efe814598f52275dee15cb70e981e1bb375e24dbf97f39788aa4c84498f33f0"),
    ("v4_phase_a_170m_seed1301.tar.gz.part01", 63233323, "c073e325d2fbe09fe4afefe75c251db59540ea3358e34d8a184d7bb2831e0f6a"),
]
PACK_ARCHIVE_SHA256 = "07f01bf4809667acc670eb9c94dfab38d28522d7bfc2d4c930e71898cff86ee7"
print({"role": WORKER_ROLE, "worker": WORKER_ID, "training_minutes": SESSION_BUDGET_MINUTES - DRAIN_RESERVE_MINUTES})

In [ ]:
# Mount the authorized account's Drive and enforce the requested accelerator.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess, torch
assert torch.cuda.is_available(), "No CUDA GPU. Select Runtime > Change runtime type > T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
assert "T4" in gpu_name.upper(), f"Expected a T4 runtime, received {gpu_name}"
assert total_gib >= 14, f"T4 memory contract failed: {total_gib:.1f} GiB"
subprocess.run(["nvidia-smi"], check=True)
print(f"READY: {gpu_name}, {total_gib:.1f} GiB")

In [ ]:
# Clone a clean operational checkout. The signed launch records the exact commit.
import pathlib, shutil, subprocess, time
REPO = pathlib.Path('/content/anra')
os.chdir('/content')  # Never delete the process's current working directory.
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
clone_command = [
    'git', '-c', 'http.version=HTTP/1.1', 'clone',
    '--depth', '1', '--single-branch', '--branch', REPO_REF,
    REPO_URL, str(REPO),
]
for attempt in range(1, 4):
    if REPO.exists():
        shutil.rmtree(REPO)
    result = subprocess.run(
        clone_command,
        text=True,
        capture_output=True,
        env=git_env,
    )
    if result.returncode == 0:
        break
    print(f'GitHub clone attempt {attempt}/3 failed (exit {result.returncode}).')
    print(result.stderr.strip() or result.stdout.strip() or 'Git returned no diagnostic.')
    if attempt == 3:
        raise RuntimeError('Unable to clone the exact training branch after 3 attempts.')
    time.sleep(2 ** attempt)
os.chdir(REPO)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert branch == REPO_REF, f'Expected branch {REPO_REF}, received {branch}'
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert not subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", "."], check=True)
print(f"Clean source branch/commit: {branch}@{commit}")

In [ ]:
# Resolve shared/current-account assets, then reconstruct the immutable 170M-token window locally.
import hashlib, tarfile
from training.colab_shared_assets import resolve_colab_training_assets

def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

SCRATCH = pathlib.Path('/content/anra-scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)
# Drive can take a few seconds to expose a newly added shortcut.
ASSETS = resolve_colab_training_assets(DRIVE_MOUNT_ROOT, discovery_timeout_seconds=45)
TRAINING_HOME = pathlib.Path(ASSETS.training_home)
VAULT_ROOT = str(ASSETS.vault_root)
os.environ['ANRA_SHARED_CHECKPOINT_DIR'] = str(TRAINING_HOME)
PACK_PATHS = {path.name: path for path in ASSETS.pack_parts}
print(f"Unified training home: {TRAINING_HOME} (step {ASSETS.vault_step})")

for name, expected_size, expected_hash in PACK_PARTS:
    mounted_part = PACK_PATHS[name]
    assert mounted_part.parent == TRAINING_HOME, f"Asset escaped training home: {mounted_part}"
    assert mounted_part.is_file(), f"Missing data pack part: {mounted_part}"
    assert mounted_part.stat().st_size == expected_size, f"Wrong size: {mounted_part}"
    assert sha256_file(mounted_part) == expected_hash, f"Corrupt data pack part: {mounted_part}"

archive = SCRATCH / 'v4_phase_a_170m_seed1301.tar.gz'
temporary = archive.with_suffix(archive.suffix + '.tmp')
with temporary.open('wb') as target:
    for name, expected_size, expected_hash in PACK_PARTS:
        part = PACK_PATHS[name]
        assert part.is_file(), f"Missing data pack part: {part}"
        assert part.stat().st_size == expected_size, f"Wrong size: {part}"
        assert sha256_file(part) == expected_hash, f"Corrupt data pack part: {part}"
        with part.open('rb') as source:
            shutil.copyfileobj(source, target, 8 * 1024 * 1024)
temporary.replace(archive)
assert sha256_file(archive) == PACK_ARCHIVE_SHA256, "Data archive hash mismatch"
pack_parent = REPO / 'output' / 'v2' / 'cloud_packs'
pack_parent.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(pack_parent, filter='data')
PACK_ROOT = pack_parent / 'v4_phase_a_170m_seed1301'
assert (PACK_ROOT / 'pack_manifest.json').is_file()
print(f"Verified data pack: {archive.stat().st_size:,} compressed bytes")

In [ ]:
# Copy the one stable, verified full-resume checkpoint to local scratch.
import json

vault_root = pathlib.Path(VAULT_ROOT)
assert vault_root == TRAINING_HOME
resume_checkpoint = SCRATCH / 'resume-source.pt'
current_checkpoint = vault_root / 'anra-v4-current-full-resume.pt'
current_metadata = vault_root / 'anra-v4-current-full-resume.json'
if current_checkpoint.is_file() and current_checkpoint.stat().st_size > 0 and current_metadata.is_file():
    source_checkpoint = current_checkpoint
    resume_step = ASSETS.vault_step
else:
    # One-time compatibility path for the old step-named Drive checkpoint.
    def portable_step(path):
        step_text = path.name[len('anra-v4-step-'):-len('-full-resume.pt')]
        return int(step_text) if step_text.isdigit() else -1
    portable = [
        path for path in vault_root.glob('anra-v4-step-*-full-resume.pt')
        if path.is_file() and path.stat().st_size > 0 and portable_step(path) >= 0
    ]
    assert portable, f'No verified full-resume checkpoint in {TRAINING_HOME}'
    portable.sort(key=portable_step, reverse=True)
    source_checkpoint = portable[0]
    resume_step = portable_step(source_checkpoint)
if source_checkpoint != current_checkpoint:
    # One-time in-place migration: rename the existing Drive object instead
    # of copying it and deleting the old 2 GB object into Drive trash.
    migration_hash = sha256_file(source_checkpoint)
    source_checkpoint.replace(current_checkpoint)
    current_metadata.write_text(json.dumps({
        'schema_version': 1, 'artifact_class': 'full_resume',
        'global_step': resume_step, 'size_bytes': current_checkpoint.stat().st_size,
        'sha256': migration_hash,
    }, sort_keys=True) + '\n', encoding='utf-8')
    source_checkpoint = current_checkpoint
source_hash = sha256_file(source_checkpoint)
# A Drive shortcut can briefly expose a completed checkpoint with stale
# pointer metadata. Verify and use the actual checkpoint bytes. Only the
# canonical publisher may replace the shared pointer under its writer lease.
if source_checkpoint == current_checkpoint:
    pointer = json.loads(current_metadata.read_text(encoding='utf-8'))
    metadata_matches = (
        int(pointer.get('size_bytes', -1)) == source_checkpoint.stat().st_size
        and str(pointer.get('sha256', '')) == source_hash
    )
    if not metadata_matches:
        print('Stale Drive pointer detected; copying the actual checkpoint bytes. Resume validation and the next protected save refresh the pointer.')
temporary = resume_checkpoint.with_suffix('.pt.tmp')
shutil.copyfile(source_checkpoint, temporary)
assert temporary.stat().st_size == source_checkpoint.stat().st_size
assert sha256_file(temporary) == source_hash
temporary.replace(resume_checkpoint)
source_label = f'portable Drive checkpoint {source_checkpoint.name}'
verified_checkpoint_hash = sha256_file(resume_checkpoint)
print(f"Verified {source_label}: step={resume_step} sha256={verified_checkpoint_hash}")

In [ ]:
# Bind the signing identity stored beside the canonical checkpoint.
key_file = ASSETS.signing_key
if key_file is None or key_file.parent != TRAINING_HOME:
    raise FileNotFoundError(f'No campaign signing key inside {TRAINING_HOME}')
private_keys = json.loads(key_file.read_text())
manifest_key = str(private_keys.get('manifest', ''))
evidence_key = str(private_keys.get('evidence', ''))
assert len(manifest_key) >= 64 and len(evidence_key) >= 64
os.environ['ANRA_MANIFEST_SIGNING_KEY'] = manifest_key
os.environ['ANRA_EVIDENCE_SIGNING_KEY'] = evidence_key
os.environ['ANRA_REQUIRE_SIGNED_EVIDENCE'] = '1'
print('Owner-private signing keys loaded without disclosure.')

In [ ]:
# Create and validate a launch bound to this commit, checkpoint, tokenizer, and remaining token window.
launch = REPO / 'output' / 'v2' / 'launch_manifests' / f'{WORKER_ID}.json'
artifact = SCRATCH / f'anra-v4-{WORKER_ID}.pt'
create_command = [
    'python', '-m', 'scripts.create_cloud_launch',
    '--pack-root', str(PACK_ROOT),
    '--output', str(launch),
    '--artifact-path', str(artifact),
    '--checkpoint-source', str(resume_checkpoint),
    '--worker-id', WORKER_ID,
    '--runtime-estimate-hours', str(SESSION_BUDGET_MINUTES / 60),
    '--batch-size', str(BATCH_SIZE),
    '--accumulation', str(GRADIENT_ACCUMULATION),
]
subprocess.run(create_command, check=True)
signed = json.loads(launch.read_text())
assert signed['git_commit'] == commit
print({
    'run_id': signed['run_id'],
    'commit': signed['git_commit'],
    'window': signed['token_window'],
    'checkpoint': signed['checkpoint_source_hash'],
})

In [ ]:
# Start the only canonical writer. New full-resume states are protected in Drive while training continues.
if WORKER_ROLE == 'verify_only':
    print('Verification complete. This worker will not modify canonical weights.')
else:
    assert WORKER_ROLE == 'canonical_trainer'
    pathlib.Path(VAULT_ROOT).mkdir(parents=True, exist_ok=True)
    os.environ['ANRA_DURABILITY_OUTBOX'] = str(SCRATCH / 'durability-outbox')
    os.environ['ANRA_DURABILITY_REPLICAS'] = json.dumps([
        {'name': 'drive-vault', 'path': VAULT_ROOT, 'kind': 'mounted_drive_single_file', 'canonical': True}
    ])
    os.environ['ANRA_DURABILITY_MIN_PROTECTED_REPLICAS'] = '1'
    os.environ['ANRA_DURABILITY_COPY_STREAMS'] = '2'
    os.environ['ANRA_DURABILITY_ACK_TIMEOUT_SECONDS'] = '1800'
    os.environ['ANRA_CHECKPOINT_EVERY_MIN'] = '60'
    os.environ['ANRA_DURABLE_CHECKPOINT_STEPS'] = '200'
    train_command = [
        'python', '-u', '-m', 'training.train_unified',
        '--mode', 'session',
        '--launch-manifest', str(launch),
        '--prepare_data', 'never',
        '--post-session-eval', 'none',
        '--data_path', 'training_data/anra_training.txt',
    ]
    process = subprocess.Popen(
        train_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=os.environ.copy(),
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Trainer exited with status {return_code}; see streamed diagnostics above.')
    print('TRAINING SESSION COMPLETE. The final protected checkpoint is in ANRA_T4_TRAINING_HOME.')